# Writing DataFrames to Files

<a href="https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week07/05.Writing-DataFrames-to-Files/notebooks/05-writing-dataframes-to-files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
Once you have cleansed, transformed, or filtered data in Pandas, you need to export your results to disk so they can be consumed by dashboards, spreadsheets, stakeholders, or web APIs.

In this notebook, you will learn how to write DataFrames to **CSV**, **JSON**, and **Excel** files, with a critical focus on the `index=False` parameter to prevent the notorious `Unnamed: 0` bug.

## 1. Load Weather Dataset
Let's load the Australian weather dataset.

In [ ]:
import pandas as pd

df = pd.read_csv('sample_weather.csv', parse_dates=['Date'])
display(df.head())

## 2. Exporting to CSV: The Crucial index=False Rule
When exporting to CSV, Pandas by default saves the integer row index (0, 1, 2...). When reloaded, Pandas creates an unwanted `Unnamed: 0` column. **Always pass `index=False` unless your index contains meaningful row labels!**

In [ ]:
# Filter hot days for a specialized report
hot_days_df = df[df['Temperature'] >= 30.0].copy()

output_csv = 'hot_weather_report.csv'
hot_days_df.to_csv(output_csv, index=False)
print(f"Exported hot weather records to '{output_csv}'")

# Verify by reading back
reloaded = pd.read_csv(output_csv)
print('Columns in reloaded CSV:', list(reloaded.columns))
display(reloaded)

## 3. Exporting to JSON for Web & API Integration
JSON is the standard format for web applications and cloud pipelines. Use `orient='records'` to generate a clean array of JSON objects.

In [ ]:
output_json = 'hot_weather_report.json'
hot_days_df.to_json(output_json, orient='records', date_format='iso', indent=2)
print(f"Exported JSON records to '{output_json}'")

# Inspect JSON preview
with open(output_json, 'r') as f:
    print(f.read()[:200] + '...')

## 4. Exporting Aggregated Summaries
Often, you summarize raw data before exporting to stakeholders.

In [ ]:
city_summary = df.groupby('City', as_index=False).agg(
    Avg_Temperature=('Temperature', 'mean'),
    Total_Rainfall=('Rainfall', 'sum')
)
summary_csv = 'city_weather_summary.csv'
city_summary.to_csv(summary_csv, index=False, float_format='%.2f')
display(pd.read_csv(summary_csv))

## 5. Exporting to Excel (Optional openpyxl)
Export DataFrames to Microsoft Excel workbooks using `to_excel()`.

In [ ]:
try:
    city_summary.to_excel('city_summary.xlsx', sheet_name='CityStats', index=False)
    print("Successfully exported Excel workbook 'city_summary.xlsx'")
except ImportError:
    print("openpyxl required for Excel export. Install via: uv add openpyxl")

## Enrichment
### Exporting Compressed CSVs on the Fly
Pandas handles compression automatically:
```python
df.to_csv('large_dataset.csv.gz', compression='gzip', index=False)
```

### Direct Markdown Output for Reports
```python
print(city_summary.to_markdown(index=False))
```

## Takeaways
- Always include `index=False` when calling `to_csv()` unless your index is a named, meaningful key.
- Use `df.to_json(..., orient='records', indent=2)` when delivering data to web services.
- Use `float_format='%.2f'` in `to_csv()` to round floating-point numbers consistently.
- Always test-read exported files to confirm columns and data formats survived the round-trip.

## Conclusion
Writing data safely and consistently ensures your data pipelines integrate cleanly with downstream systems, colleagues, and external clients.

## Exercises
**Exercise 1:** Filter records for Sydney only and export them to `sydney_weather.csv` with `index=False`.

**Exercise 2:** Export `city_summary` to a JSON file named `city_summary.json` with `orient='records'`.

**Exercise 3:** Reload `sydney_weather.csv` and verify it has no `Unnamed: 0` column.

In [ ]:
# Write your practice code here

# --- Solutions ---
# sydney_df = df[df['City'] == 'Sydney']
# sydney_df.to_csv('sydney_weather.csv', index=False)
#
# city_summary.to_json('city_summary.json', orient='records', indent=2)
#
# recheck = pd.read_csv('sydney_weather.csv')
# print('Columns:', list(recheck.columns))
# print('Has Unnamed:', any('Unnamed' in col for col in recheck.columns))